
# Notebook 13 — Quantum ESPRESSO exact-state pilot preparation

**Project:** Physics-constrained and dependency-aware machine learning for transferable prediction of insertion-electrode properties
**Pilot case:** `Na_candidate_06` (`NaCoPCO7 → Na3CoPCO7`)
**DFT engine:** Quantum ESPRESSO 7.5 in Ubuntu 24.04 under WSL2

## Purpose

This notebook converts the exact, provenance-controlled structures from Notebooks 14B/14C into a safe Quantum ESPRESSO pilot package. It:

- verifies the exact charged and discharged structures and their hashes;
- verifies the balanced Na-insertion reaction and formula-unit normalization;
- checks the installed QE executable, MPI, and the five selected PBE pseudopotentials;
- creates targeted low-cost SCF inputs for the charged endpoint, discharged endpoint, and bcc Na;
- creates magnetic-starting-state, cutoff, k-point, rank-benchmark, `vc-relax`, and PBE+U-sensitivity inputs;
- generates local WSL Bash runners with strict gates;
- deploys the package to WSL when possible;
- **does not automatically launch DFT calculations**.

## Required files beside this notebook

1. `Notebook 11*.zip`
2. `Notebook 12*.zip`

The notebook accepts renamed copies such as `Notebook 11(1).zip`.

## Safety policy

- No VASP `POTCAR` files are read or copied.
- No pseudopotential file is embedded in the generated package.
- Expensive relaxations are blocked until targeted smoke tests pass.
- PBE+U is a sensitivity branch, not the primary baseline.


In [ ]:

from pathlib import Path
import csv
import hashlib
import json
import math
import os
import platform
import re
import shlex
import shutil
import subprocess
import tarfile
import time
import zipfile
from datetime import datetime, timezone

# ========================= USER CONFIGURATION =========================

INPUT_DIR = Path.cwd()

# Leave as None for automatic discovery.
NOTEBOOK_11_ZIP = None
NOTEBOOK_12_ZIP = None

CANDIDATE_ID = "Na_candidate_06"
WSL_DISTRO = "Ubuntu-24.04"
WSL_HOME = str(Path.home())
WSL_CMT_ROOT = str(
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
WSL_QE_ROOT = str(
    Path(WSL_CMT_ROOT)
    / "software"
    / "qe-7.5"
)
WSL_PW_X = f"{WSL_QE_ROOT}/bin/pw.x"
WSL_PP_X = f"{WSL_QE_ROOT}/bin/pp.x"
WSL_PSEUDO_DIR = f"{WSL_CMT_ROOT}/pseudopotentials/SSSP_1.3.0_PBE_selected"
WSL_DEPLOY_DIR = f"{WSL_CMT_ROOT}/notebook_13_qe"

RUN_WSL_PREFLIGHT = os.environ.get("CMT_13_TEST_MODE", "0") != "1"
DEPLOY_TO_WSL = os.environ.get("CMT_13_TEST_MODE", "0") != "1"
RUN_DFT_FROM_NOTEBOOK = False   # Keep False. DFT is launched manually in WSL.

# Fast, targeted smoke settings. These are not final production settings.
SMOKE_ECUTWFC_RY = 50
SMOKE_ECUTRHO_RY = 400
SMOKE_RANKS = 4

# Convergence ladders generated by this notebook; they are not run automatically.
CUTOFF_WFC_GRID_RY = [60, 80, 100]
CUTOFF_RHO_RATIO = 8

# PBE+U is generated only as a controlled sensitivity branch.
GENERATE_PBEU_SENSITIVITY = True
CO_U_SENSITIVITY_EV = 3.32

OUTPUT_ROOT = INPUT_DIR / "notebook_13_qe"
WORK_ROOT = INPUT_DIR / "work/quantum_espresso_pilot"

print("Notebook 13 configuration loaded.")
print("INPUT_DIR:", INPUT_DIR.resolve())
print("RUN_WSL_PREFLIGHT:", RUN_WSL_PREFLIGHT)
print("DEPLOY_TO_WSL:", DEPLOY_TO_WSL)
print("RUN_DFT_FROM_NOTEBOOK:", RUN_DFT_FROM_NOTEBOOK)


In [ ]:

# Utilities: hashing, safe ZIP extraction, input discovery, POSCAR parsing, and WSL execution.

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def safe_extract_zip(zip_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            member_path = (destination / member.filename).resolve()
            try:
                member_path.relative_to(destination)
            except ValueError as exc:
                raise RuntimeError(f"Unsafe ZIP member: {member.filename}") from exc
        zf.extractall(destination)


def find_input_zip(kind: str, explicit=None, required_member_suffix: str | None = None) -> Path:
    def zip_has_required_member(path: Path) -> bool:
        if required_member_suffix is None:
            return True
        suffix = required_member_suffix.replace("\\", "/")
        try:
            with zipfile.ZipFile(path, "r") as zf:
                return any(name.replace("\\", "/").endswith(suffix) for name in zf.namelist())
        except (zipfile.BadZipFile, OSError):
            return False

    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        if not zip_has_required_member(path):
            raise RuntimeError(
                f"{path} does not contain the required member ending in "
                f"{required_member_suffix!r}."
            )
        return path

    patterns = [
        f"*{kind}*.zip",
        f"*{kind.lower()}*.zip",
        f"*{kind.upper()}*.zip",
    ]
    roots = [INPUT_DIR, Path.home() / "Downloads", Path.home() / "Desktop"]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            candidates.extend(root.glob(pattern))

    candidates = [
        p.resolve() for p in candidates
        if p.is_file()
        and "14D" not in p.name
        and "PRIVATE_PILOT_DFT_PACKAGE" not in p.name
        and zip_has_required_member(p)
    ]
    candidates = sorted(set(candidates), key=lambda p: (p.stat().st_mtime, p.stat().st_size), reverse=True)
    if not candidates:
        raise FileNotFoundError(
            f"Could not find a ZIP containing '{kind}' and required member "
            f"{required_member_suffix!r}. Put the correct ZIP beside this notebook "
            "or set the corresponding path in the configuration cell."
        )
    return candidates[0]


def find_unique(root: Path, relative_suffix: str) -> Path:
    suffix_norm = relative_suffix.replace("\\", "/")
    matches = [
        p for p in root.rglob(Path(relative_suffix).name)
        if p.as_posix().endswith(suffix_norm)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one file ending in '{relative_suffix}', found {len(matches)}: {matches}"
        )
    return matches[0]


def read_csv_rows(path: Path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def det3(m):
    return (
        m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
    )


def parse_poscar(path: Path):
    raw_lines = [line.rstrip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(raw_lines) < 8:
        raise ValueError(f"POSCAR too short: {path}")

    comment = raw_lines[0].strip()
    scale = float(raw_lines[1].split()[0])
    if scale <= 0:
        raise ValueError("This notebook requires a positive POSCAR scale factor.")

    lattice = []
    for i in range(2, 5):
        vec = [float(x) * scale for x in raw_lines[i].split()[:3]]
        if len(vec) != 3:
            raise ValueError(f"Invalid lattice line in {path}: {raw_lines[i]}")
        lattice.append(vec)

    symbol_tokens = raw_lines[5].split()
    count_tokens = raw_lines[6].split()
    if not all(re.fullmatch(r"[A-Za-z][A-Za-z0-9]*", x) for x in symbol_tokens):
        raise ValueError("VASP 4 POSCAR without element symbols is not supported.")
    counts = [int(x) for x in count_tokens]
    if len(symbol_tokens) != len(counts):
        raise ValueError("POSCAR symbols/counts mismatch.")

    cursor = 7
    selective = raw_lines[cursor].lower().startswith("s")
    if selective:
        cursor += 1

    coordinate_mode = raw_lines[cursor].strip().lower()
    cursor += 1
    nat = sum(counts)

    coords = []
    labels = []
    expanded = []
    for symbol, count in zip(symbol_tokens, counts):
        expanded.extend([symbol] * count)

    for i in range(nat):
        tokens = raw_lines[cursor + i].split()
        xyz = [float(v) for v in tokens[:3]]
        if coordinate_mode.startswith(("c", "k")):
            xyz = [v * scale for v in xyz]
        coords.append(xyz)
        labels.append(tokens[3] if len(tokens) >= 4 and re.fullmatch(r"[A-Za-z][A-Za-z0-9]*", tokens[3]) else expanded[i])

    volume = abs(det3(lattice))
    return {
        "comment": comment,
        "symbols": symbol_tokens,
        "counts": counts,
        "nat": nat,
        "lattice": lattice,
        "coordinate_mode": "crystal" if coordinate_mode.startswith("d") else "angstrom",
        "coords": coords,
        "labels": labels,
        "volume": volume,
        "source_path": str(path),
        "source_sha256": sha256_file(path),
    }


def reduced_formula_text(symbols, counts):
    parts = []
    for s, n in zip(symbols, counts):
        parts.append(s if n == 1 else f"{s}{n}")
    return "".join(parts)


def is_wsl_runtime() -> bool:
    return platform.system() == "Linux" and "microsoft" in platform.release().lower()


def run_wsl(command: str, timeout: int = 90):
    if os.environ.get("CMT_13_TEST_MODE", "0") == "1":
        return {"status": "SKIPPED_TEST_MODE", "returncode": None, "stdout": "", "stderr": ""}

    if os.name == "nt":
        argv = ["wsl.exe", "-d", WSL_DISTRO, "bash", "-lc", command]
    elif is_wsl_runtime():
        argv = ["bash", "-lc", command]
    else:
        return {
            "status": "SKIPPED_NON_WSL_HOST",
            "returncode": None,
            "stdout": "",
            "stderr": "Notebook is neither Windows-with-WSL nor a WSL runtime.",
        }

    try:
        cp = subprocess.run(argv, capture_output=True, text=True, timeout=timeout, check=False)
        return {
            "status": "EXECUTED",
            "returncode": cp.returncode,
            "stdout": cp.stdout,
            "stderr": cp.stderr,
        }
    except subprocess.TimeoutExpired as exc:
        return {
            "status": "TIMEOUT",
            "returncode": 124,
            "stdout": exc.stdout or "",
            "stderr": exc.stderr or "",
        }


print("Utility functions ready.")


In [ ]:

# Discover and extract Notebook 11/14C packages.

ZIP_14B = find_input_zip("14B", NOTEBOOK_11_ZIP, "Notebook 11/processed/11_exact_structure_manifest.csv")
ZIP_14C = find_input_zip("14C", NOTEBOOK_12_ZIP, "Notebook 12/processed/12_pilot_reaction_balance.csv")

print("Using 14B ZIP:", ZIP_14B)
print("Using 14C ZIP:", ZIP_14C)
print("14B ZIP SHA256:", sha256_file(ZIP_14B))
print("14C ZIP SHA256:", sha256_file(ZIP_14C))

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)

EXTRACT_14B = WORK_ROOT / "input_14B"
EXTRACT_14C = WORK_ROOT / "input_14C"
EXTRACT_14B.mkdir()
EXTRACT_14C.mkdir()

safe_extract_zip(ZIP_14B, EXTRACT_14B)
safe_extract_zip(ZIP_14C, EXTRACT_14C)

MANIFEST_14B = find_unique(EXTRACT_14B, "Notebook 11/processed/11_exact_structure_manifest.csv")
CASE_MANIFEST_14B = find_unique(EXTRACT_14B, "Notebook 11/processed/11_hypothesis_driven_dft_case_manifest.csv")
REACTION_14C = find_unique(EXTRACT_14C, "Notebook 12/processed/12_pilot_reaction_balance.csv")
DECISION_14B = find_unique(EXTRACT_14B, "Notebook 11/metadata/11_final_decision.json")
DECISION_14C = find_unique(EXTRACT_14C, "Notebook 12/metadata/12_final_decision.json")

print("Input extraction complete.")


In [ ]:

# Validate exact-state provenance, formulas, hashes, volumes, and reaction balance.

EXPECTED_STRUCTURE_JSON_HASHES = {
    "charged": "fb7360829e590bf36a0cff51d76612c09f062ae1a4e8f88380494083e44bea47",
    "discharged": "3deed4cbf2de11c20a5ab345f0b9330dd737ac4606b09d06d54486aa5fd81bdf",
}
EXPECTED_POSCAR_HASHES = {
    "charged": "e01cd0d560eddac9052a271c99cded040846ee90d7dd3f3057f20d1ee92a1eb2",
    "discharged": "78f7fee5b2fe66d4680046a8bfc5eafd80d56880e57ed977ee03f1b9138595cb",
}

rows14b = read_csv_rows(MANIFEST_14B)
candidate_rows = [r for r in rows14b if r["candidate_id"] == CANDIDATE_ID]
if len(candidate_rows) != 2:
    raise RuntimeError(f"Expected two exact-state rows for {CANDIDATE_ID}, found {len(candidate_rows)}")

case_rows = [r for r in read_csv_rows(CASE_MANIFEST_14B) if r["candidate_id"] == CANDIDATE_ID]
if len(case_rows) != 1:
    raise RuntimeError("Candidate case manifest row is missing or duplicated.")

reaction_rows = [r for r in read_csv_rows(REACTION_14C) if r["candidate_id"] == CANDIDATE_ID]
if len(reaction_rows) != 1:
    raise RuntimeError("Candidate reaction-balance row is missing or duplicated.")
reaction = reaction_rows[0]

decision14b = json.loads(DECISION_14B.read_text(encoding="utf-8"))
decision14c = json.loads(DECISION_14C.read_text(encoding="utf-8"))

exact_paths = {}
structures = {}
provenance_audit = []

for state in ("charged", "discharged"):
    row = next(r for r in candidate_rows if r["state_role"] == state)
    poscar = find_unique(
        EXTRACT_14C,
        f"Notebook 12/pilot_dft_package/{CANDIDATE_ID}/{state}/00_exact_structure/POSCAR",
    )
    structure_json = find_unique(
        EXTRACT_14C,
        f"Notebook 12/pilot_dft_package/{CANDIDATE_ID}/{state}/00_exact_structure/structure.json",
    )

    actual_poscar_hash = sha256_file(poscar)
    actual_structure_hash = sha256_file(structure_json)
    manifest_structure_hash = row["structure_sha256"]

    parsed = parse_poscar(poscar)
    structures[state] = parsed
    exact_paths[state] = {"poscar": poscar, "structure_json": structure_json}

    expected_formula = "Na2Co2P2C2O14" if state == "charged" else "Na12Co4P4C4O28"
    actual_formula = reduced_formula_text(parsed["symbols"], parsed["counts"])

    checks = {
        "candidate_id": CANDIDATE_ID,
        "state": state,
        "endpoint_id": row["endpoint_id"],
        "manifest_structure_hash": manifest_structure_hash,
        "actual_structure_json_hash": actual_structure_hash,
        "expected_structure_json_hash": EXPECTED_STRUCTURE_JSON_HASHES[state],
        "actual_poscar_hash": actual_poscar_hash,
        "expected_poscar_hash": EXPECTED_POSCAR_HASHES[state],
        "formula": actual_formula,
        "expected_cell_formula": expected_formula,
        "nat": parsed["nat"],
        "volume_A3": parsed["volume"],
        "structure_hash_pass": (
            actual_structure_hash == EXPECTED_STRUCTURE_JSON_HASHES[state]
            == manifest_structure_hash
        ),
        "poscar_hash_pass": actual_poscar_hash == EXPECTED_POSCAR_HASHES[state],
        "formula_pass": actual_formula == expected_formula,
    }
    checks["all_pass"] = all(
        checks[k] for k in ("structure_hash_pass", "poscar_hash_pass", "formula_pass")
    )
    provenance_audit.append(checks)

# Formula-unit and volume checks from Notebook 12.
charged_fu = float(reaction["charged_cell_host_formula_units"])
discharged_fu = float(reaction["discharged_cell_host_formula_units"])
charged_volume_per_fu = structures["charged"]["volume"] / charged_fu
discharged_volume_per_fu = structures["discharged"]["volume"] / discharged_fu
volume_change = discharged_volume_per_fu / charged_volume_per_fu - 1.0

reaction_checks = {
    "balanced_reaction": reaction["balanced_reaction"],
    "delta_na_per_host_formula_unit": float(reaction["delta_na_per_host_formula_unit"]),
    "charged_volume_per_fu_A3": charged_volume_per_fu,
    "discharged_volume_per_fu_A3": discharged_volume_per_fu,
    "computed_signed_volume_change": volume_change,
    "manifest_signed_volume_change": float(reaction["signed_volume_change_fraction_from_exact_input_structures"]),
    "reaction_pass": reaction["balanced_reaction"] == "charged_host + 2 Na(metal) -> discharged_host",
    "delta_na_pass": abs(float(reaction["delta_na_per_host_formula_unit"]) - 2.0) < 1e-12,
    "volume_match_pass": abs(
        volume_change - float(reaction["signed_volume_change_fraction_from_exact_input_structures"])
    ) < 1e-6,
}

if not all(row["all_pass"] for row in provenance_audit):
    raise RuntimeError("Exact-state provenance validation failed.")
if not all(reaction_checks[k] for k in ("reaction_pass", "delta_na_pass", "volume_match_pass")):
    raise RuntimeError("Reaction-balance validation failed.")

print("Exact-state provenance: PASS")
for row in provenance_audit:
    print(row["state"], row["endpoint_id"], row["formula"], row["nat"], f"{row['volume_A3']:.6f} A^3")
print("Reaction:", reaction_checks["balanced_reaction"])
print("Exact-input volume change:", f"{100*volume_change:.4f}%")


In [ ]:

# Read-only WSL/QE/pseudopotential preflight.

PSEUDO_POLICY = {
    "Na": {
        "filename": "na_pbe_v1.5.uspp.F.UPF",
        "sha256": "f84820c4280d21603bd50ae76a08e15c26864391ea055cf50affc4aedefc8f87",
        "mass": 22.98976928,
    },
    "Co": {
        "filename": "Co_pbe_v1.2.uspp.F.UPF",
        "sha256": "a5f392a0d35fc78cc9e2ac1636bb1cd02e9b80a063ea368e85be83b0e4523c45",
        "mass": 58.933194,
    },
    "P": {
        "filename": "P.pbe-n-rrkjus_psl.1.0.0.UPF",
        "sha256": "2d112dfec2e2d9b75a971574d9116aa92249988177791659bbcd2ba15f23c20c",
        "mass": 30.973761998,
    },
    "C": {
        "filename": "C.pbe-n-kjpaw_psl.1.0.0.UPF",
        "sha256": "9900d1efd50b9848e31849f39094b33348486b400ee51e0f3922f716137cf3d7",
        "mass": 12.011,
    },
    "O": {
        "filename": "O.pbe-n-kjpaw_psl.0.1.UPF",
        "sha256": "7c4b6ed541f83d0afdf5c1d3a8c611340f073f3e2b11e95b7963bb2ee26929aa",
        "mass": 15.999,
    },
}

preflight_rows = []

if RUN_WSL_PREFLIGHT:
    command_lines = [
        "set -u",
        f"test -x {shlex.quote(WSL_PW_X)} && echo 'PW_X|PASS|{WSL_PW_X}' || echo 'PW_X|FAIL|missing'",
        f"test -x {shlex.quote(WSL_PP_X)} && echo 'PP_X|PASS|{WSL_PP_X}' || echo 'PP_X|FAIL|missing'",
        "command -v mpirun >/dev/null && echo \"MPI|PASS|$(command -v mpirun)\" || echo 'MPI|FAIL|missing'",
        "echo \"MPI_VERSION|INFO|$(mpirun -version 2>/dev/null | head -n 1)\"",
        "echo \"LSCPU|INFO|$(lscpu 2>/dev/null | grep -E '^CPU\\(s\\):|^Thread\\(s\\) per core:|^Core\\(s\\) per socket:|^Socket\\(s\\):' | tr '\\n' ';')\"",
    ]
    for element, meta in PSEUDO_POLICY.items():
        path = f"{WSL_PSEUDO_DIR}/{meta['filename']}"
        expected = meta["sha256"]
        command_lines.append(
            f"if test -s {shlex.quote(path)}; then "
            f"actual=$(sha256sum {shlex.quote(path)} | awk '{{print $1}}'); "
            f"if test \"$actual\" = {shlex.quote(expected)}; then "
            f"echo {shlex.quote('PSEUDO_'+element)}'|PASS|'\"$actual\"; "
            f"else echo {shlex.quote('PSEUDO_'+element)}'|FAIL|'\"$actual\"; fi; "
            f"else echo {shlex.quote('PSEUDO_'+element)}'|FAIL|missing'; fi"
        )
    command_lines.append("timeout 20s mpirun -np 2 hostname >/dev/null 2>&1 && echo 'MPI_SMOKE|PASS|2 ranks' || echo 'MPI_SMOKE|FAIL|unable to launch 2 ranks'")
    result = run_wsl("\n".join(command_lines), timeout=120)

    if result["status"] == "EXECUTED":
        for line in result["stdout"].splitlines():
            parts = line.split("|", 2)
            if len(parts) == 3:
                preflight_rows.append({"check": parts[0], "status": parts[1], "detail": parts[2]})
        if result["stderr"].strip():
            preflight_rows.append({"check": "WSL_STDERR", "status": "INFO", "detail": result["stderr"].strip()})
    else:
        preflight_rows.append({"check": "WSL_EXECUTION", "status": "SKIPPED", "detail": result["status"]})
else:
    preflight_rows.append({"check": "WSL_PREFLIGHT", "status": "SKIPPED", "detail": "RUN_WSL_PREFLIGHT=False or test mode"})

required_preflight_checks = {"PW_X", "PP_X", "MPI", "MPI_SMOKE"} | {f"PSEUDO_{x}" for x in PSEUDO_POLICY}
statuses = {r["check"]: r["status"] for r in preflight_rows}
wsl_preflight_pass = all(statuses.get(name) == "PASS" for name in required_preflight_checks)

print("WSL preflight rows:")
for row in preflight_rows:
    print(row)
print("WSL preflight overall:", "PASS" if wsl_preflight_pass else "HOLD/SKIPPED")


In [ ]:

# Generate the complete QE pilot package. No DFT is launched by this cell.

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

for directory in [
    OUTPUT_ROOT / "audit",
    OUTPUT_ROOT / "metadata",
    OUTPUT_ROOT / "processed",
    OUTPUT_ROOT / "qe_pilot" / "config",
    OUTPUT_ROOT / "qe_pilot" / "results",
]:
    directory.mkdir(parents=True, exist_ok=True)

PILOT_ROOT = OUTPUT_ROOT / "qe_pilot"

def write_csv(path: Path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def species_configuration(structure, magnetic_variant="fm_medium"):
    base_labels = list(structure["labels"])
    if magnetic_variant != "afm_split":
        species_order = list(structure["symbols"])
        atom_labels = base_labels
        starts = {"Co": {"fm_low": 0.20, "fm_medium": 0.60, "fm_high": 0.90}.get(magnetic_variant, 0.60)}
        return species_order, atom_labels, starts, False

    # Alternating Co sublattices; nosym is required to preserve opposite starts.
    atom_labels = []
    co_index = 0
    for label in base_labels:
        if label == "Co":
            atom_labels.append("CoA" if co_index % 2 == 0 else "CoB")
            co_index += 1
        else:
            atom_labels.append(label)
    species_order = []
    for label in atom_labels:
        if label not in species_order:
            species_order.append(label)
    starts = {"CoA": 0.60, "CoB": -0.60}
    return species_order, atom_labels, starts, True


def pseudo_element(species_label):
    if species_label in ("CoA", "CoB"):
        return "Co"
    return species_label


def qe_input_text(
    structure,
    calculation,
    prefix,
    ecutwfc,
    ecutrho,
    kmesh,
    magnetic_variant=None,
    degauss=0.02,
    hubbard_u_ev=None,
):
    magnetic = magnetic_variant is not None and "Co" in structure["symbols"]
    if magnetic:
        species_order, atom_labels, starts, nosym = species_configuration(structure, magnetic_variant)
    else:
        species_order = list(structure["symbols"])
        atom_labels = list(structure["labels"])
        starts, nosym = {}, False

    control = [
        "&CONTROL",
        f"  calculation = '{calculation}',",
        f"  prefix = '{prefix}',",
        f"  pseudo_dir = '{WSL_PSEUDO_DIR}',",
        "  outdir = './tmp',",
        "  verbosity = 'high',",
        "  disk_io = 'low',",
        "  tstress = .true.,",
        "  tprnfor = .true.,",
    ]
    if calculation in ("relax", "vc-relax"):
        control += [
            "  nstep = 200,",
            "  etot_conv_thr = 1.0d-5,",
            "  forc_conv_thr = 1.0d-3,",
        ]
    control.append("/")

    system = [
        "&SYSTEM",
        "  ibrav = 0,",
        f"  nat = {structure['nat']},",
        f"  ntyp = {len(species_order)},",
        f"  ecutwfc = {float(ecutwfc):.1f},",
        f"  ecutrho = {float(ecutrho):.1f},",
        "  input_dft = 'PBE',",
        "  occupations = 'smearing',",
        "  smearing = 'mv',",
        f"  degauss = {float(degauss):.5f},",
    ]
    if magnetic:
        system.append("  nspin = 2,")
        for idx, species in enumerate(species_order, start=1):
            if species in starts:
                system.append(f"  starting_magnetization({idx}) = {starts[species]:.6f},")
    if nosym:
        system.append("  nosym = .true.,")
    system.append("/")

    electrons = [
        "&ELECTRONS",
        "  conv_thr = 1.0d-8,",
        "  electron_maxstep = 200,",
        "  mixing_mode = 'plain',",
        "  mixing_beta = 0.30,",
        "  diagonalization = 'david',",
        "/",
    ]

    ions = []
    cell = []
    if calculation in ("relax", "vc-relax"):
        ions = [
            "&IONS",
            "  ion_dynamics = 'bfgs',",
            "/",
        ]
    if calculation == "vc-relax":
        cell = [
            "&CELL",
            "  cell_dynamics = 'bfgs',",
            "  press = 0.0,",
            "  press_conv_thr = 0.5,",
            "  cell_dofree = 'all',",
            "/",
        ]

    species_lines = ["ATOMIC_SPECIES"]
    for species in species_order:
        element = pseudo_element(species)
        meta = PSEUDO_POLICY[element]
        species_lines.append(f"{species:<3s} {meta['mass']:.9f} {meta['filename']}")

    cell_lines = ["CELL_PARAMETERS angstrom"]
    cell_lines.extend("  " + " ".join(f"{x: .16f}" for x in vec) for vec in structure["lattice"])

    pos_lines = [f"ATOMIC_POSITIONS {structure['coordinate_mode']}"]
    for label, coord in zip(atom_labels, structure["coords"]):
        pos_lines.append(f"{label:<3s} " + " ".join(f"{x: .16f}" for x in coord))

    k_lines = [
        "K_POINTS automatic",
        f"{kmesh[0]} {kmesh[1]} {kmesh[2]} 0 0 0",
    ]

    hubbard_lines = []
    if hubbard_u_ev is not None:
        hubbard_lines = ["HUBBARD (ortho-atomic)"]
        co_species = [s for s in species_order if pseudo_element(s) == "Co"]
        for s in co_species:
            hubbard_lines.append(f"U {s}-3d {float(hubbard_u_ev):.6f}")

    all_lines = control + [""] + system + [""] + electrons
    if ions:
        all_lines += [""] + ions
    if cell:
        all_lines += [""] + cell
    all_lines += [""] + species_lines + [""] + cell_lines + [""] + pos_lines + [""] + k_lines
    if hubbard_lines:
        all_lines += [""] + hubbard_lines
    return "\n".join(all_lines) + "\n"


def write_job(relpath, structure, calculation, prefix, ecutwfc, ecutrho, kmesh,
              magnetic_variant=None, hubbard_u_ev=None, role=None):
    job_dir = PILOT_ROOT / relpath
    job_dir.mkdir(parents=True, exist_ok=True)
    (job_dir / "tmp").mkdir(exist_ok=True)
    input_path = job_dir / "pw.in"
    input_path.write_text(
        qe_input_text(
            structure=structure,
            calculation=calculation,
            prefix=prefix,
            ecutwfc=ecutwfc,
            ecutrho=ecutrho,
            kmesh=kmesh,
            magnetic_variant=magnetic_variant,
            hubbard_u_ev=hubbard_u_ev,
        ),
        encoding="utf-8",
        newline="\n",
    )
    return {
        "job_directory": str(relpath).replace("\\", "/"),
        "input_file": str((relpath / "pw.in")).replace("\\", "/"),
        "role": role or calculation,
        "calculation": calculation,
        "state": relpath.parts[1] if len(relpath.parts) > 1 else "",
        "magnetic_variant": magnetic_variant or "nonmagnetic",
        "hubbard_u_ev": "" if hubbard_u_ev is None else hubbard_u_ev,
        "ecutwfc_Ry": ecutwfc,
        "ecutrho_Ry": ecutrho,
        "kmesh": "x".join(map(str, kmesh)),
        "input_sha256": sha256_file(input_path),
    }


# Exact bcc Na reference inherited from Notebook 12.
NA_POSCAR = find_unique(
    EXTRACT_14C,
    "Notebook 12/pilot_dft_package/Na_metal_reference/01_relax/POSCAR",
)
structures["na_metal"] = parse_poscar(NA_POSCAR)

# Copy exact inputs for independent audit.
exact_copy_root = PILOT_ROOT / "00_exact_structures"
for state in ("charged", "discharged"):
    dst = exact_copy_root / state
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(exact_paths[state]["poscar"], dst / "POSCAR")
    shutil.copy2(exact_paths[state]["structure_json"], dst / "structure.json")
na_dst = exact_copy_root / "na_metal"
na_dst.mkdir(parents=True, exist_ok=True)
shutil.copy2(NA_POSCAR, na_dst / "POSCAR")

calculation_manifest = []

# 1. Targeted smoke inputs.
calculation_manifest.append(write_job(
    Path("01_smoke") / "charged" / "pbe_fm_medium",
    structures["charged"], "scf", "c06_chg_smoke",
    SMOKE_ECUTWFC_RY, SMOKE_ECUTRHO_RY, (1, 1, 1),
    magnetic_variant="fm_medium", role="targeted_smoke",
))
calculation_manifest.append(write_job(
    Path("01_smoke") / "discharged" / "pbe_fm_medium",
    structures["discharged"], "scf", "c06_dis_smoke",
    SMOKE_ECUTWFC_RY, SMOKE_ECUTRHO_RY, (1, 1, 1),
    magnetic_variant="fm_medium", role="targeted_smoke",
))
calculation_manifest.append(write_job(
    Path("01_smoke") / "na_metal" / "pbe",
    structures["na_metal"], "scf", "na_smoke",
    SMOKE_ECUTWFC_RY, SMOKE_ECUTRHO_RY, (6, 6, 6),
    role="targeted_smoke",
))

# 2. Rank benchmark on the smaller charged endpoint.
for ranks in (4, 12, 24):
    calculation_manifest.append(write_job(
        Path("02_rank_benchmark") / f"ranks_{ranks:02d}",
        structures["charged"], "scf", f"c06_rank_{ranks}",
        SMOKE_ECUTWFC_RY, SMOKE_ECUTRHO_RY, (1, 1, 1),
        magnetic_variant="fm_medium", role=f"rank_benchmark_{ranks}",
    ))

# 3. Magnetic-starting-state screen.
baseline_meshes = {
    "charged": (3, 4, 2),
    "discharged": (3, 3, 2),
    "na_metal": (12, 12, 12),
}
for state in ("charged", "discharged"):
    for variant in ("fm_low", "fm_medium", "fm_high", "afm_split"):
        calculation_manifest.append(write_job(
            Path("03_magnetism") / state / variant,
            structures[state], "scf", f"c06_{state[:3]}_{variant}",
            80, 640, baseline_meshes[state],
            magnetic_variant=variant, role="magnetic_screen",
        ))

# 4. Cutoff convergence.
for state in ("charged", "discharged", "na_metal"):
    for ecut in CUTOFF_WFC_GRID_RY:
        calculation_manifest.append(write_job(
            Path("04_cutoff") / state / f"ecut_{ecut:03d}",
            structures[state], "scf", f"c06_{state[:3]}_ec{ecut}",
            ecut, ecut * CUTOFF_RHO_RATIO, baseline_meshes[state],
            magnetic_variant="fm_medium" if state != "na_metal" else None,
            role="cutoff_convergence",
        ))

# 5. K-point convergence at 80/640 Ry.
kmesh_grid = {
    "charged": [(2, 3, 1), (3, 4, 2), (4, 5, 3)],
    "discharged": [(2, 2, 1), (3, 3, 2), (4, 4, 3)],
    "na_metal": [(8, 8, 8), (12, 12, 12), (16, 16, 16)],
}
for state, meshes in kmesh_grid.items():
    for mesh in meshes:
        tag = "x".join(map(str, mesh))
        calculation_manifest.append(write_job(
            Path("05_kpoint") / state / f"k_{tag}",
            structures[state], "scf", f"c06_{state[:3]}_k{tag.replace('x','')}",
            80, 640, mesh,
            magnetic_variant="fm_medium" if state != "na_metal" else None,
            role="kpoint_convergence",
        ))

# 6. PBE vc-relax templates. These are hard-gated by the runner.
for state in ("charged", "discharged", "na_metal"):
    calculation_manifest.append(write_job(
        Path("06_vc_relax_pbe") / state,
        structures[state], "vc-relax", f"c06_{state[:3]}_vcr",
        80, 640, baseline_meshes[state],
        magnetic_variant="fm_medium" if state != "na_metal" else None,
        role="pbe_vc_relax_template",
    ))

# 7. Controlled PBE+U sensitivity templates (not primary evidence).
if GENERATE_PBEU_SENSITIVITY:
    for state in ("charged", "discharged"):
        calculation_manifest.append(write_job(
            Path("07_pbeu_sensitivity") / state / f"U_{CO_U_SENSITIVITY_EV:.2f}",
            structures[state], "scf", f"c06_{state[:3]}_u",
            80, 640, baseline_meshes[state],
            magnetic_variant="fm_medium",
            hubbard_u_ev=CO_U_SENSITIVITY_EV,
            role="pbeu_sensitivity_template",
        ))

# Write policy/config files.
config = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "candidate_id": CANDIDATE_ID,
    "wsl_distro": WSL_DISTRO,
    "wsl_qe_root": WSL_QE_ROOT,
    "wsl_pw_x": WSL_PW_X,
    "wsl_pp_x": WSL_PP_X,
    "wsl_pseudo_dir": WSL_PSEUDO_DIR,
    "smoke_ranks": SMOKE_RANKS,
    "smoke_ecutwfc_Ry": SMOKE_ECUTWFC_RY,
    "smoke_ecutrho_Ry": SMOKE_ECUTRHO_RY,
    "run_dft_from_notebook": RUN_DFT_FROM_NOTEBOOK,
    "pbeu_sensitivity_enabled": GENERATE_PBEU_SENSITIVITY,
    "co_u_sensitivity_eV": CO_U_SENSITIVITY_EV,
    "scientific_policy": {
        "primary_baseline": "spin-polarized PBE",
        "pbeu_role": "sensitivity only; not tuned to reproduce database voltage",
        "static_energy_policy": "final SCF only after successful cell/ion relaxation",
        "external_claim": "independent cross-code exact-state spot check, not reproduction of VASP energies",
    },
}
(PILOT_ROOT / "config" / "13_qe_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)
(PILOT_ROOT / "config" / "pseudopotential_policy.json").write_text(
    json.dumps(PSEUDO_POLICY, indent=2), encoding="utf-8"
)
(PILOT_ROOT / "config" / "reaction_balance.json").write_text(
    json.dumps(reaction_checks, indent=2), encoding="utf-8"
)

write_csv(
    OUTPUT_ROOT / "audit" / "13_input_provenance_audit.csv",
    provenance_audit,
)
write_csv(
    OUTPUT_ROOT / "audit" / "13_wsl_qe_preflight_audit.csv",
    preflight_rows,
    fieldnames=["check", "status", "detail"],
)
write_csv(
    OUTPUT_ROOT / "processed" / "configuration/13_calculation_manifest.csv",
    calculation_manifest,
)

pseudo_audit_rows = [
    {
        "element": element,
        "filename": meta["filename"],
        "expected_sha256": meta["sha256"],
        "wsl_path": f"{WSL_PSEUDO_DIR}/{meta['filename']}",
        "not_embedded_in_package": True,
    }
    for element, meta in PSEUDO_POLICY.items()
]
write_csv(
    OUTPUT_ROOT / "audit" / "13_pseudopotential_policy_audit.csv",
    pseudo_audit_rows,
)

expected_result_schema = [
    {"field": "job_id", "unit": "", "required": True},
    {"field": "state", "unit": "", "required": True},
    {"field": "method", "unit": "", "required": True},
    {"field": "converged_electronic", "unit": "boolean", "required": True},
    {"field": "job_done", "unit": "boolean", "required": True},
    {"field": "total_energy_Ry", "unit": "Ry/cell", "required": True},
    {"field": "total_energy_eV", "unit": "eV/cell", "required": True},
    {"field": "total_magnetization", "unit": "Bohr magneton/cell", "required": False},
    {"field": "absolute_magnetization", "unit": "Bohr magneton/cell", "required": False},
    {"field": "final_volume_A3", "unit": "A^3/cell", "required": False},
    {"field": "wall_seconds", "unit": "s", "required": False},
    {"field": "error_flag", "unit": "", "required": True},
]
write_csv(
    OUTPUT_ROOT / "processed" / "configuration/13_expected_result_schema.csv",
    expected_result_schema,
)

print(f"Generated {len(calculation_manifest)} QE calculation inputs.")


In [ ]:

# Generate strict local-WSL runner scripts and the run-order README.

preflight_script = r"""#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
RESULTS="$ROOT/results"
mkdir -p "$RESULTS"

PW="${QE_PW:-$(command -v pw.x || true)}"
PP="${QE_PP:-$(command -v pp.x || true)}"
PSEUDO="${QE_PSEUDO_DIR:-$ROOT/pseudopotentials}"
POLICY="$ROOT/config/configuration/pseudopotential_policy.json"
REPORT="$RESULTS/00_preflight_report.txt"

: > "$REPORT"
fail=0

check() {
    local name="$1"
    shift
    if "$@"; then
        echo "$name|PASS" | tee -a "$REPORT"
    else
        echo "$name|FAIL" | tee -a "$REPORT"
        fail=1
    fi
}

check PW_X test -x "$PW"
check PP_X test -x "$PP"
check MPIRUN bash -lc 'command -v mpirun >/dev/null'
check JQ bash -lc 'command -v jq >/dev/null'

while IFS=$'\t' read -r element filename expected
do
    [[ "$element" == "element" ]] && continue
    file="$PSEUDO/$filename"
    if [[ ! -s "$file" ]]; then
        echo "PSEUDO_${element}|FAIL|missing:$file" | tee -a "$REPORT"
        fail=1
        continue
    fi
    actual="$(sha256sum "$file" | awk '{print $1}')"
    if [[ "$actual" == "$expected" ]]; then
        echo "PSEUDO_${element}|PASS|$actual" | tee -a "$REPORT"
    else
        echo "PSEUDO_${element}|FAIL|expected=$expected actual=$actual" | tee -a "$REPORT"
        fail=1
    fi
done < <(
    {
        echo -e "element\tfilename\texpected"
        jq -r 'to_entries[] | [.key, .value.filename, .value.sha256] | @tsv' "$POLICY"
    }
)

input_count="$(find "$ROOT" -type f -name pw.in | wc -l)"
if [[ "$input_count" -gt 0 ]]; then
    echo "QE_INPUT_COUNT|PASS|$input_count" | tee -a "$REPORT"
else
    echo "QE_INPUT_COUNT|FAIL|0" | tee -a "$REPORT"
    fail=1
fi

if timeout 20s mpirun -np 2 hostname >/dev/null 2>&1; then
    echo "MPI_LAUNCH_2|PASS" | tee -a "$REPORT"
else
    echo "MPI_LAUNCH_2|FAIL" | tee -a "$REPORT"
    fail=1
fi

if grep -R -nE 'REPLACE_ME|UNCONFIRMED|6\.x\.x|SITE_SPECIFIC' "$ROOT" \
    -include='*.in' -include='*.sh' -include='*.json' >/dev/null 2>&1
then
    echo "PLACEHOLDER_SCAN|FAIL" | tee -a "$REPORT"
    fail=1
else
    echo "PLACEHOLDER_SCAN|PASS" | tee -a "$REPORT"
fi

if [[ "$fail" -ne 0 ]]; then
    echo "PREFLIGHT_HOLD" | tee -a "$REPORT"
    exit 2
fi

echo "PREFLIGHT_PASS" | tee -a "$REPORT"
"""

smoke_script = r"""#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PW="${QE_PW:-$(command -v pw.x || true)}"
RANKS="${SMOKE_RANKS:-4}"
RESULTS="$ROOT/results"
SUMMARY="$RESULTS/01_smoke_summary.tsv"
mkdir -p "$RESULTS"

bash "$ROOT/00_preflight.sh"

printf "job\treturn_code\tconverged\tjob_done\terror_flag\tenergy_Ry\n" > "$SUMMARY"

run_job() {
    local rel="$1"
    local dir="$ROOT/$rel"
    local out="$dir/pw.out"
    local err="$dir/pw.err"

    rm -rf "$dir/tmp"
    mkdir -p "$dir/tmp"

    set +e
    (
      cd "$dir"
      export OMP_NUM_THREADS=1
      export OPENBLAS_NUM_THREADS=1
      export MKL_NUM_THREADS=1
      export OMP_STACKSIZE=512m
      ulimit -s unlimited
      timeout 4h mpirun -bind-to core -map-by core -np "$RANKS" \
          "$PW" -in pw.in > pw.out 2> pw.err
    )
    rc=$?
    set -e

    converged=0
    job_done=0
    error_flag=0
    grep -q "convergence has been achieved" "$out" && converged=1 || true
    grep -q "JOB DONE" "$out" && job_done=1 || true
    if grep -Eiq "Error in routine|NaN|MPI_ABORT|segmentation|stopping" "$out" "$err"; then
        error_flag=1
    fi
    energy="$(awk '/^!/ {e=$5} END {print e}' "$out")"

    printf "%s\t%s\t%s\t%s\t%s\t%s\n" \
        "$rel" "$rc" "$converged" "$job_done" "$error_flag" "${energy:-NA}" \
        | tee -a "$SUMMARY"

    [[ "$rc" -eq 0 && "$converged" -eq 1 && "$job_done" -eq 1 && "$error_flag" -eq 0 ]]
}

fail=0
run_job "01_smoke/charged/pbe_fm_medium" || fail=1
run_job "01_smoke/discharged/pbe_fm_medium" || fail=1
run_job "01_smoke/na_metal/pbe" || fail=1

tar -czf "$RESULTS/13_targeted_smoke_results.tar.gz" \
    -C "$ROOT" \
    results/00_preflight_report.txt \
    results/01_smoke_summary.tsv \
    01_smoke/charged/pbe_fm_medium/pw.in \
    01_smoke/charged/pbe_fm_medium/pw.out \
    01_smoke/charged/pbe_fm_medium/pw.err \
    01_smoke/discharged/pbe_fm_medium/pw.in \
    01_smoke/discharged/pbe_fm_medium/pw.out \
    01_smoke/discharged/pbe_fm_medium/pw.err \
    01_smoke/na_metal/pbe/pw.in \
    01_smoke/na_metal/pbe/pw.out \
    01_smoke/na_metal/pbe/pw.err

if [[ "$fail" -ne 0 ]]; then
    echo "TARGETED_SMOKE_HOLD"
    exit 3
fi

echo "TARGETED_SMOKE_PASS"
echo "Upload: $RESULTS/13_targeted_smoke_results.tar.gz"
"""

rank_script = r"""#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PW="${QE_PW:-$(command -v pw.x || true)}"
RESULTS="$ROOT/results"
SUMMARY="$RESULTS/02_rank_benchmark.tsv"
mkdir -p "$RESULTS"

grep -q "TARGETED_SMOKE_PASS" <(bash "$ROOT/01_check_smoke_only.sh") || {
    echo "HOLD: smoke tests have not passed."
    exit 3
}

printf "ranks\treturn_code\twall_seconds\tconverged\tjob_done\n" > "$SUMMARY"

for ranks in 4 12 24
do
    dir="$ROOT/02_rank_benchmark/ranks_$(printf '%02d' "$ranks")"
    rm -rf "$dir/tmp"
    mkdir -p "$dir/tmp"
    start="$(date +%s)"

    set +e
    (
      cd "$dir"
      export OMP_NUM_THREADS=1
      export OPENBLAS_NUM_THREADS=1
      export MKL_NUM_THREADS=1
      ulimit -s unlimited
      timeout 2h mpirun -bind-to core -map-by core -np "$ranks" \
          "$PW" -in pw.in > pw.out 2> pw.err
    )
    rc=$?
    set -e

    stop="$(date +%s)"
    wall="$((stop-start))"
    converged=0
    done_flag=0
    grep -q "convergence has been achieved" "$dir/pw.out" && converged=1 || true
    grep -q "JOB DONE" "$dir/pw.out" && done_flag=1 || true
    printf "%s\t%s\t%s\t%s\t%s\n" "$ranks" "$rc" "$wall" "$converged" "$done_flag" \
        | tee -a "$SUMMARY"
done

echo "RANK_BENCHMARK_COMPLETE"
"""

check_smoke_script = r"""#!/usr/bin/env bash
set -euo pipefail
ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
summary="$ROOT/results/01_smoke_summary.tsv"
if [[ ! -f "$summary" ]]; then
    echo "TARGETED_SMOKE_HOLD"
    exit 1
fi
bad="$(awk -F'\t' 'NR>1 && !($2==0 && $3==1 && $4==1 && $5==0) {n++} END {print n+0}' "$summary")"
if [[ "$bad" -eq 0 && "$(awk 'END{print NR}' "$summary")" -eq 4 ]]; then
    echo "TARGETED_SMOKE_PASS"
else
    echo "TARGETED_SMOKE_HOLD"
    exit 1
fi
"""

generic_batch_script = r"""#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PW="${QE_PW:-$(command -v pw.x || true)}"
RANKS="${QE_RANKS:-12}"
STAGE="${1:-}"
RESULTS="$ROOT/results"

case "$STAGE" in
  magnetism) stage_dir="03_magnetism" ;;
  cutoff) stage_dir="04_cutoff" ;;
  kpoint) stage_dir="05_kpoint" ;;
  *) echo "Usage: bash 03_run_screen.sh {magnetism|cutoff|kpoint}"; exit 2 ;;
esac

bash "$ROOT/01_check_smoke_only.sh" >/dev/null || {
    echo "HOLD: targeted smoke tests must pass first."
    exit 3
}

summary="$RESULTS/dft_validation/${stage_dir}_summary.tsv"
printf "job\treturn_code\tconverged\tjob_done\tenergy_Ry\n" > "$summary"

while IFS= read -r input
do
    dir="$(dirname "$input")"
    rel="${dir#$ROOT/}"
    rm -rf "$dir/tmp"
    mkdir -p "$dir/tmp"

    set +e
    (
      cd "$dir"
      export OMP_NUM_THREADS=1
      export OPENBLAS_NUM_THREADS=1
      export MKL_NUM_THREADS=1
      ulimit -s unlimited
      timeout 12h mpirun -bind-to core -map-by core -np "$RANKS" \
          "$PW" -in pw.in > pw.out 2> pw.err
    )
    rc=$?
    set -e

    converged=0
    done_flag=0
    grep -q "convergence has been achieved" "$dir/pw.out" && converged=1 || true
    grep -q "JOB DONE" "$dir/pw.out" && done_flag=1 || true
    energy="$(awk '/^!/ {e=$5} END {print e}' "$dir/pw.out")"
    printf "%s\t%s\t%s\t%s\t%s\n" "$rel" "$rc" "$converged" "$done_flag" "${energy:-NA}" \
        | tee -a "$summary"
done < <(find "$ROOT/$stage_dir" -type f -name pw.in | sort)

tar -czf "$RESULTS/13_${stage_dir}_results.tar.gz" \
    -C "$ROOT" "$stage_dir" "results/$(basename "$summary")"

echo "${STAGE^^}_SCREEN_COMPLETE"
echo "Upload: $RESULTS/13_${stage_dir}_results.tar.gz"
"""

relax_script = r"""#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PW="${QE_PW:-$(command -v pw.x || true)}"
RANKS="${QE_RANKS:-12}"
APPROVAL="$ROOT/config/APPROVE_PBE_VC_RELAX"

bash "$ROOT/01_check_smoke_only.sh" >/dev/null || {
    echo "HOLD: targeted smoke tests must pass first."
    exit 3
}

if [[ ! -f "$APPROVAL" ]] || [[ "$(tr -d '[:space:]' < "$APPROVAL")" != "APPROVED" ]]; then
    echo "HOLD: vc-relax is blocked."
    echo "After reviewing smoke, rank, magnetism, cutoff, and k-point results, run:"
    echo "echo APPROVED > $APPROVAL"
    exit 4
fi

for input in \
  "$ROOT/06_vc_relax_pbe/charged/pw.in" \
  "$ROOT/06_vc_relax_pbe/discharged/pw.in" \
  "$ROOT/06_vc_relax_pbe/na_metal/pw.in"
do
    dir="$(dirname "$input")"
    rm -rf "$dir/tmp"
    mkdir -p "$dir/tmp"
    (
      cd "$dir"
      export OMP_NUM_THREADS=1
      export OPENBLAS_NUM_THREADS=1
      export MKL_NUM_THREADS=1
      ulimit -s unlimited
      timeout 72h mpirun -bind-to core -map-by core -np "$RANKS" \
          "$PW" -in pw.in > pw.out 2> pw.err
    )
    grep -q "JOB DONE" "$dir/pw.out"
    grep -q "End final coordinates" "$dir/pw.out"
done

echo "PBE_VC_RELAX_COMPLETE"
"""

pbeu_script = r"""#!/usr/bin/env bash
set -euo pipefail
ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
echo "PBE+U is a sensitivity branch only."
echo "It is intentionally not run by Notebook 13."
echo "First complete and review the PBE workflow."
echo "Then create: $ROOT/config/APPROVE_PBEU_SENSITIVITY"
[[ -f "$ROOT/config/APPROVE_PBEU_SENSITIVITY" ]] || exit 4
echo "Approval file found. Run individual PBE+U jobs only after method review."
"""

readme_text = f"""# Notebook 13 Quantum ESPRESSO pilot

## Current scope

Candidate: `{CANDIDATE_ID}`
Reaction: `NaCoPCO7 + 2 Na(metal) -> Na3CoPCO7`

This package contains no pseudopotential files and launches nothing automatically.

## Run order in Ubuntu WSL

```bash
cd {WSL_DEPLOY_DIR}/qe_pilot
bash 00_preflight.sh
bash 01_run_targeted_smoke.sh
```

After `TARGETED_SMOKE_PASS`, upload:

```text
{WSL_DEPLOY_DIR}/qe_pilot/results/13_targeted_smoke_results.tar.gz
```

Do not start relaxation yet.

## Later stages, only after reviewing prior outputs

```bash
bash 02_run_rank_benchmark.sh
bash 03_run_screen.sh magnetism
bash 03_run_screen.sh cutoff
bash 03_run_screen.sh kpoint
```

`04_run_pbe_vc_relax.sh` is blocked until the file
`config/APPROVE_PBE_VC_RELAX` contains exactly `APPROVED`.

PBE+U is a sensitivity branch and has a separate approval gate.
"""

scripts = {
    "00_preflight.sh": preflight_script,
    "01_run_targeted_smoke.sh": smoke_script,
    "01_check_smoke_only.sh": check_smoke_script,
    "02_run_rank_benchmark.sh": rank_script,
    "03_run_screen.sh": generic_batch_script,
    "04_run_pbe_vc_relax.sh": relax_script,
    "05_pbeu_sensitivity_gate.sh": pbeu_script,
}

for filename, text in scripts.items():
    path = PILOT_ROOT / filename
    path.write_text(text, encoding="utf-8", newline="\n")
    try:
        path.chmod(0o755)
    except OSError:
        pass

(PILOT_ROOT / "README_RUN_ORDER.md").write_text(readme_text, encoding="utf-8", newline="\n")

# Explicitly create empty approval instructions, not approval files.
(PILOT_ROOT / "config" / "APPROVAL_POLICY.txt").write_text(
    "Do not create approval files until all preceding results have been reviewed.\n",
    encoding="utf-8",
)

print("WSL runner scripts generated.")
print("First WSL command after deployment:")
print(f"cd {WSL_DEPLOY_DIR}/qe_pilot && bash 00_preflight.sh")


In [ ]:

# Build audits, output manifest, package ZIP, and optionally deploy to WSL.

# Go/no-go gates.
gates = [
    {
        "gate": "notebook14B_exact_state_hashes",
        "pass": all(row["all_pass"] for row in provenance_audit),
        "detail": "charged/discharged structure.json and POSCAR hashes verified",
    },
    {
        "gate": "notebook14C_reaction_balance",
        "pass": all(reaction_checks[k] for k in ("reaction_pass", "delta_na_pass", "volume_match_pass")),
        "detail": reaction_checks["balanced_reaction"],
    },
    {
        "gate": "qe_inputs_generated",
        "pass": len(calculation_manifest) >= 30,
        "detail": f"{len(calculation_manifest)} inputs",
    },
    {
        "gate": "licensed_potcar_absent",
        "pass": not any(p.name == "POTCAR" for p in OUTPUT_ROOT.rglob("*") if p.is_file()),
        "detail": "No VASP POTCAR copied",
    },
    {
        "gate": "pseudopotentials_not_embedded",
        "pass": not any(p.suffix.lower() == ".upf" for p in OUTPUT_ROOT.rglob("*") if p.is_file()),
        "detail": "Only filenames and expected SHA256 values recorded",
    },
    {
        "gate": "automatic_dft_disabled",
        "pass": RUN_DFT_FROM_NOTEBOOK is False,
        "detail": "Manual WSL launch required",
    },
    {
        "gate": "wsl_qe_preflight",
        "pass": wsl_preflight_pass,
        "detail": "PASS required for direct deployment decision; may be SKIPPED in test mode",
    },
]
write_csv(OUTPUT_ROOT / "audit" / "13_go_no_go_gate_audit.csv", gates)

all_scientific_gates = all(g["pass"] for g in gates if g["gate"] != "wsl_qe_preflight")
decision = (
    "GO_TO_TARGETED_QE_SMOKE_TESTS"
    if all_scientific_gates and wsl_preflight_pass
    else "PACKAGE_READY_WSL_PREFLIGHT_REQUIRED"
    if all_scientific_gates
    else "HOLD_13_INPUT_OR_PACKAGE_FAILURE"
)

decision_payload = {
    "decision": decision,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "candidate_id": CANDIDATE_ID,
    "automatic_dft_launched": False,
    "next_action": (
        f"cd {WSL_DEPLOY_DIR}/qe_pilot && bash 00_preflight.sh && bash 01_run_targeted_smoke.sh"
        if decision == "GO_TO_TARGETED_QE_SMOKE_TESTS"
        else "Resolve failed preflight gates, rerun Notebook 13, and do not launch DFT."
    ),
}
(OUTPUT_ROOT / "metadata" / "13_final_decision.json").write_text(
    json.dumps(decision_payload, indent=2), encoding="utf-8"
)

# Output file manifest, excluding the manifest itself until after collection.
manifest_rows = []
for path in sorted(p for p in OUTPUT_ROOT.rglob("*") if p.is_file()):
    rel = path.relative_to(OUTPUT_ROOT).as_posix()
    if rel == "metadata/13_output_file_manifest.csv":
        continue
    manifest_rows.append({
        "relative_path": rel,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })
write_csv(
    OUTPUT_ROOT / "metadata" / "13_output_file_manifest.csv",
    manifest_rows,
)

# ZIP package.
ZIP_OUTPUT = INPUT_DIR / "13_quantum_espresso_exact_state_pilot_package.zip"
if ZIP_OUTPUT.exists():
    ZIP_OUTPUT.unlink()
with zipfile.ZipFile(ZIP_OUTPUT, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(p for p in OUTPUT_ROOT.rglob("*") if p.is_file()):
        zf.write(path, arcname=(Path("notebook_13_qe") / path.relative_to(OUTPUT_ROOT)).as_posix())

# Tar stream for WSL deployment.
TAR_OUTPUT = INPUT_DIR / "13_quantum_espresso_exact_state_pilot_package.tar.gz"
if TAR_OUTPUT.exists():
    TAR_OUTPUT.unlink()
with tarfile.open(TAR_OUTPUT, "w:gz") as tf:
    tf.add(OUTPUT_ROOT, arcname="notebook_13_qe")

deployment = {
    "attempted": False,
    "status": "SKIPPED",
    "detail": "",
}

if DEPLOY_TO_WSL and os.environ.get("CMT_13_TEST_MODE", "0") != "1":
    deployment["attempted"] = True
    if os.name == "nt":
        deploy_command = (
            f"set -e; parent={shlex.quote(str(Path(WSL_DEPLOY_DIR).parent))}; "
            f"target={shlex.quote(WSL_DEPLOY_DIR)}; "
            f"mkdir -p \"$parent\"; "
            f"if [ -d \"$target\" ]; then mv \"$target\" \"${{target}}_backup_$(date +%Y%m%d_%H%M%S)\"; fi; "
            f"tar -xzf - -C \"$parent\""
        )
        cp = subprocess.run(
            ["wsl.exe", "-d", WSL_DISTRO, "bash", "-lc", deploy_command],
            input=TAR_OUTPUT.read_bytes(),
            capture_output=True,
            timeout=180,
            check=False,
        )
        deployment["status"] = "PASS" if cp.returncode == 0 else "FAIL"
        deployment["detail"] = (cp.stderr or cp.stdout).decode(errors="replace") if isinstance(cp.stderr or cp.stdout, bytes) else str(cp.stderr or cp.stdout)
    elif is_wsl_runtime():
        target = Path(WSL_DEPLOY_DIR)
        if target.exists():
            backup = target.with_name(target.name + "_backup_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
            target.rename(backup)
        shutil.copytree(OUTPUT_ROOT, target)
        deployment["status"] = "PASS"
        deployment["detail"] = str(target)
    else:
        deployment["status"] = "SKIPPED_NON_WSL_HOST"
        deployment["detail"] = "Copy the generated package into WSL manually."

(OUTPUT_ROOT / "metadata" / "13_deployment_status.json").write_text(
    json.dumps(deployment, indent=2), encoding="utf-8"
)

print("=" * 72)
print("Notebook 13 FINAL DECISION:", decision)
print("QE inputs generated:", len(calculation_manifest))
print("ZIP package:", ZIP_OUTPUT.resolve())
print("ZIP SHA256:", sha256_file(ZIP_OUTPUT))
print("Deployment:", deployment)
print("=" * 72)

if decision == "GO_TO_TARGETED_QE_SMOKE_TESTS" and deployment.get("status") == "PASS":
    print("NEXT COMMANDS IN UBUNTU:")
    print(f"cd {WSL_DEPLOY_DIR}/qe_pilot")
    print("bash 00_preflight.sh")
    print("bash 01_run_targeted_smoke.sh")
elif decision == "GO_TO_TARGETED_QE_SMOKE_TESTS":
    print("Package is valid, but deployment was not completed.")
    print("Copy the generated notebook_13_qe folder into WSL, then run the README commands.")
else:
    print("Do not run DFT. Review 13_go_no_go_gate_audit.csv and the WSL preflight audit.")



## Interpretation of the final decision

### `GO_TO_TARGETED_QE_SMOKE_TESTS`

The exact-state inputs, pseudopotential hashes, QE executable, MPI, and WSL deployment all passed. Open Ubuntu and run only:

```bash
cd ~/cmt-dft/notebook_13_qe/qe_pilot
bash 00_preflight.sh
bash 01_run_targeted_smoke.sh
```

Upload the generated:

```text
~/cmt-dft/notebook_13_qe/qe_pilot/results/13_targeted_smoke_results.tar.gz
```

### `PACKAGE_READY_WSL_PREFLIGHT_REQUIRED`

The scientific package was generated, but this notebook could not verify WSL directly. Do not run DFT until `00_preflight.sh` reports `PREFLIGHT_PASS`.

### `HOLD_13_INPUT_OR_PACKAGE_FAILURE`

At least one structure, reaction, hash, or package-integrity gate failed. Do not proceed.
